In [15]:
'''
pip install langchain-huggingface sentence-transformers
pip install chromadb

Instantiation Pipeline:
1) Full-body validated base corpus 
2) User can choose to add local, specific pdf files
2) all pdfs are cleaned and added with metadata stored for citations
3) Chunking is performed such taht semantic structrue is maintained without duplicate data
4) Chunks are embedded into vectorized database with metadata stored
'''

import os, re
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from pypdf import PdfReader


def pdf_cleaner(paths):
    """Take a list of PDF links, return {filename: full_cleaned_text}."""
    docs = {}
    for path in paths:
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            raw = page.extract_text() or ""
            raw = re.sub(r"-\n", "", raw)          # join hyphenated breaks
            text += re.sub(r"\s+", " ", raw).strip() + " "
        docs[os.path.basename(path)] = text.strip()
    return docs
    
def add_documents(old_documents, new_documents):
    """Merge new {filename: text} into the existing dict, skipping duplicates."""
    combined = dict(old_documents)
    for filename, text in new_documents.items():
        if filename in combined:
            print(f"skipping (already present): {filename}")
            continue
        combined[filename] = text
    return combined

def semantic_chunker(documents):
    """documents: {filename: text} -> list of chunk Documents."""
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    text_splitter = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=70.0)

    all_chunks = []
    for filename, text in documents.items():
        chunks = text_splitter.create_documents(
            [text],
            metadatas=[{"filename": filename}])
        for chunk_index, chunk in enumerate(chunks):
            chunk.metadata["chunk_index"] = chunk_index
        all_chunks.extend(chunks)
    return all_chunks

def chunk_embedder(all_chunks, persist_dir="./index"):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(
        all_chunks,
        embeddings,
        persist_directory=persist_dir)   # written to disk so you don't re-embed
    return vectorstore

#Instantiation code
list_of_pdfs = [] #user writes a list of filepaths they'd like to add
base_corpus = {} #placeholder


list_of_documents = add_documents(base_corpus, pdf_cleaner(list_of_pdfs))
chunks = semantic_chunker(list_of_documents)
vectorized_database = chunk_embedder(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18950.45it/s]


ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in upsert.

In [ ]:

'''
Query Pipeline:
1.1) pre-filtering deterministic lookup bypass
1.2) misc pre-filtering
2) Query embedding (choose: Exact query, Hypothetical Document Embedding, Multi-Query Union)
3) Top-ksearch
3.1) re-ranking top-k
3.2) similarity lower bound threshold refusal trigger
4) chunk token embedding
4.1) LLM-RAG system prompt
'''